In [ ]:
"""
NOTEBOOK 1: CRASH DETECTOR
==========================
Detecta crashes, fatals, ANRs en logs STB/STV

INPUT: Log file (.txt)
OUTPUT: Reporte de crashes detectados
"""

import pandas as pd
import re
from datetime import datetime
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

LOG_FILE = "ruta/al/log.txt"  # ← CAMBIAR AQUÍ
OUTPUT_DIR = "outputs"

# Patterns de eventos críticos
CRITICAL_PATTERNS = {
    'crash': r'crash|CRASH|crashed',
    'anr': r'ANR|not responding',
    'fatal': r'fatal|FATAL',
    'restart': r'restart.*crashed|Scheduling restart',
    'force_close': r'Force.*close|Force.*stop'
}

# ============================================================================
# FUNCIONES
# ============================================================================

def parse_line(line: str) -> dict | None:
    if not line or line.startswith('-----'):
        return None
    
    try:
        # Formato A: MM-DD HH:MM:SS.mmm PID TID E Tag: mensaje
        pattern_a = r'^(\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2}\.\d{3})\s+(\d+)\s+(\d+)\s+([VDIWEF])\s+([^:]+):\s+(.+)$'
        match = re.match(pattern_a, line)
        
        if match:
            return {
                "date": match.group(1),
                "time": match.group(2),
                "pid": match.group(3),
                "tid": match.group(4),
                "level": match.group(5),
                "tag": match.group(6).strip(),
                "message": match.group(7)
            }
        
        # Formato B: MM-DD HH:MM:SS.mmm E/Tag(PID): mensaje
        pattern_b = r'^(\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2}\.\d{3})\s+([VDIWEF])/([^(]+)\(\s*(\d+)\):\s+(.+)$'
        match = re.match(pattern_b, line)
        
        if match:
            return {
                "date": match.group(1),
                "time": match.group(2),
                "level": match.group(3),
                "tag": match.group(4).strip(),
                "pid": match.group(5).strip(),
                "tid": match.group(5).strip(),
                "message": match.group(6)
            }
        
        return None
    except:
        return None


def time_rounded(parsed_line: str) -> str:
    time_minutes = datetime.strptime(parsed_line, "%H:%M:%S.%f")
    time_minutes = time_minutes.replace(second=0, microsecond=0).strftime("%H:%M")
    return time_minutes


def read_log(filepath: str):
    parsed_logs = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            parsed = parse_line(line)
            if parsed:
                parsed["time"] = time_rounded(parsed["time"])
                parsed_logs.append(parsed)
    return parsed_logs


def detect_crashes(df):
    """Detecta eventos críticos en el log"""
    
    events = []
    
    # Buscar cada patrón
    for event_type, pattern in CRITICAL_PATTERNS.items():
        matches = df[df['message'].str.contains(pattern, case=False, regex=True, na=False)]
        
        for _, row in matches.iterrows():
            events.append({
                'type': event_type,
                'minute': row['time'],
                'level': row['level'],
                'tag': row['tag'],
                'message': row['message'][:200]
            })
    
    return pd.DataFrame(events)


def detect_fatals(df):
    """Detecta fatals en el log"""
    fatals = df[df['level'] == 'F'].copy()
    
    if len(fatals) == 0:
        return None
    
    return fatals[['time', 'tag', 'message']]


def detect_error_spikes(df):
    """Detecta picos de errors por minuto"""
    
    by_minute = df.groupby('time').agg({
        'level': lambda x: (x == 'E').sum(),
        'tag': 'count'
    }).rename(columns={'level': 'errors', 'tag': 'total'})
    
    by_minute['error_rate'] = (by_minute['errors'] / by_minute['total'] * 100).round(1)
    
    # Picos > 30%
    spikes = by_minute[by_minute['error_rate'] > 30].copy()
    spikes = spikes.sort_values('error_rate', ascending=False)
    
    return spikes


def main():
    print("="*80)
    print("🚨 CRASH DETECTOR")
    print("="*80)
    
    if not Path(LOG_FILE).exists():
        print(f"❌ Archivo no encontrado: {LOG_FILE}")
        return
    
    print(f"\n📂 Procesando: {LOG_FILE}")
    
    # Parse
    parsed_logs = read_log(LOG_FILE)
    
    if len(parsed_logs) == 0:
        print("❌ No se pudo parsear el log")
        return
    
    df = pd.DataFrame(parsed_logs)
    
    print(f"✅ Logs parseados: {len(df):,}")
    
    # Detectar eventos críticos
    print("\n" + "="*80)
    print("🔍 EVENTOS CRÍTICOS DETECTADOS")
    print("="*80)
    
    events = detect_crashes(df)
    
    if len(events) == 0:
        print("✅ No se detectaron crashes, ANRs o eventos críticos")
    else:
        print(f"\n⚠️ TOTAL: {len(events)} eventos críticos\n")
        
        for event_type in events['type'].unique():
            type_events = events[events['type'] == event_type]
            print(f"\n🔴 {event_type.upper()}: {len(type_events)} eventos")
            
            for _, evt in type_events.head(5).iterrows():
                print(f"   [{evt['minute']}] {evt['tag']}: {evt['message'][:100]}")
    
    # Detectar fatals
    print("\n" + "="*80)
    print("💀 FATALS DETECTADOS")
    print("="*80)
    
    fatals = detect_fatals(df)
    
    if fatals is None:
        print("✅ No se detectaron Fatals")
    else:
        print(f"\n⚠️ TOTAL: {len(fatals)} fatals\n")
        for _, fatal in fatals.head(10).iterrows():
            print(f"   [{fatal['time']}] {fatal['tag']}: {fatal['message'][:100]}")
    
    # Detectar picos
    print("\n" + "="*80)
    print("📈 PICOS DE ERRORS DETECTADOS")
    print("="*80)
    
    spikes = detect_error_spikes(df)
    
    if len(spikes) == 0:
        print("✅ No se detectaron picos de errors (>30%)")
    else:
        print(f"\n⚠️ TOTAL: {len(spikes)} minutos con picos\n")
        print(spikes.head(10).to_string())
    
    # Resumen final
    print("\n" + "="*80)
    print("🎯 RESUMEN")
    print("="*80)
    
    has_crash = len(events) > 0
    has_fatal = fatals is not None
    has_spike = len(spikes) > 0
    
    severity = "🟢 NORMAL"
    
    if has_fatal or (has_crash and has_spike):
        severity = "🔴 CRÍTICO - CRASH DETECTADO"
    elif has_crash or has_spike:
        severity = "🟡 ADVERTENCIA - DEGRADACIÓN DETECTADA"
    
    print(f"\nEstado: {severity}")
    print(f"Eventos críticos: {len(events)}")
    print(f"Fatals: {len(fatals) if fatals is not None else 0}")
    print(f"Picos de errors: {len(spikes)}")
    
    # Guardar resultados
    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    
    if len(events) > 0:
        events.to_csv(f"{OUTPUT_DIR}/crash_events.csv", index=False)
        print(f"\n✅ Eventos guardados: {OUTPUT_DIR}/crash_events.csv")
    
    if fatals is not None:
        fatals.to_csv(f"{OUTPUT_DIR}/fatals.csv", index=False)
        print(f"✅ Fatals guardados: {OUTPUT_DIR}/fatals.csv")
    
    if len(spikes) > 0:
        spikes.to_csv(f"{OUTPUT_DIR}/error_spikes.csv")
        print(f"✅ Picos guardados: {OUTPUT_DIR}/error_spikes.csv")


if __name__ == "__main__":
    main()